In [3]:
"""
Simple Examples of Single-Hop and Double-Hop Teleportation
Easy to understand and test
"""

import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, state_fidelity, DensityMatrix,partial_trace
from qiskit_aer import AerSimulator


# ============================================================================
# EXAMPLE 1: Single-Hop Quantum Controlled Teleportation
# ============================================================================

def single_hop_teleportation_example():
    """
    Single-Hop QCT: Alice → Bob (with Charlie's control)
    
    Qubits:
      q[0] - Alice's input qubit
      q[1] - Charlie's control qubit  
      q[2] - Intermediate qubit
      q[3] - Bob's output qubit
    """
    
    print("\n" + "="*70)
    print("SINGLE-HOP QUANTUM CONTROLLED TELEPORTATION")
    print("="*70)
    
    # Parameters for input state: |φ⟩ = cos(θ)|0⟩ + e^(iφ)sin(θ)|1⟩
    theta = np.pi/4  # 45 degrees
    phi = 0          # No phase
    
    print(f"\nInput state: |φ⟩ = cos({theta:.3f})|0⟩ + e^(i·{phi}π)sin({theta:.3f})|1⟩")
    print(f"           = {np.cos(theta):.3f}|0⟩ + {np.sin(theta):.3f}|1⟩")
    
    # Create circuit
    qr = QuantumRegister(4, 'q')
    cr = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(qr, cr)
    
    # Step 1: Prepare Alice's input state
    qc.ry(2*theta, 0)  # Apply rotation
    qc.rz(phi, 0)
    qc.barrier(label='Input')
    
    # Step 2: Create GHZ state for quantum channel
    # GHZ state: (|000⟩ + |111⟩)/√2 on qubits [1,2,3]
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(1, 3)
    qc.barrier(label='GHZ')
    
    # Step 3: Apply inverse GHZ transformation on [0,1,2]
    qc.cx(0, 2)
    qc.cx(0, 1)
    qc.h(0)
    qc.barrier(label='Transform')
    
    # Step 4: Measure Alice (q[0]) and Charlie (q[1])
    qc.measure(0, 0)
    qc.measure(1, 1)
    qc.barrier(label='Measure')
    
    # Step 5: Apply corrections to Bob's qubit (q[3])    
    # qc.z(3).c_if(cr[0], 1)# Z if Alice measured 1
    # qc.x(3).c_if(cr[1], 1)# X if Charlie measured 1 
    with qc.if_test((cr[0], 1)):
        qc.z(3)
    with qc.if_test((cr[1], 1)):
        qc.x(3)

    
    print("\nCircuit created:")
    print(f"  Qubits: {qc.num_qubits}")
    print(f"  Depth: {qc.depth()}")
    print(f"  Gates: {sum(qc.count_ops().values())}")
    
    # Simulate
    qc.draw()
    qc.save_statevector()
    simulator = AerSimulator(method="statevector")
    result = simulator.run(qc).result()
    final_state = result.get_statevector()

    # Extract Bob's qubit only
    rho = DensityMatrix(final_state)
    rho_bob = partial_trace(rho, [0, 1, 2])

    # Target state
    target = Statevector([
        np.cos(theta),
        np.exp(1j * phi) * np.sin(theta)
    ])

    fidelity = state_fidelity(rho_bob, target)
    
    print(f"\nFidelity: {fidelity:.6f}")
    print(f"Status: {'✓ SUCCESS' if fidelity > 0.999 else '✗ FAILED'}")
    
    return qc, fidelity


# ============================================================================
# EXAMPLE 2: Double-Hop Quantum Controlled Teleportation
# ============================================================================

def double_hop_teleportation_example():
    """
    Double-Hop QCT: Alice → Node1 → Bob (with 2 Charlies)
    
    Qubits:
      q[0] - Alice's input qubit
      q[1] - Charlie1's control qubit
      q[2] - Intermediate hop 1
      q[3] - Node1 output / Hop2 input
      q[4] - Charlie2's control qubit
      q[5] - Intermediate hop 2
      q[6] - Bob's output qubit
    """
    
    print("\n" + "="*70)
    print("DOUBLE-HOP QUANTUM CONTROLLED TELEPORTATION")
    print("="*70)
    
    # Parameters for input state
    theta = np.pi/4
    phi = 0
    
    print(f"\nInput state: |φ⟩ = cos({theta:.3f})|0⟩ + e^(i·{phi}π)sin({theta:.3f})|1⟩")
    print(f"           = {np.cos(theta):.3f}|0⟩ + {np.sin(theta):.3f}|1⟩")
    
    # Create circuit
    qr = QuantumRegister(7, 'q')
    cr = ClassicalRegister(4, 'c')
    qc = QuantumCircuit(qr, cr)
    
    # Step 1: Prepare Alice's input state (q[0])
    qc.ry(2*theta, 0)
    qc.rz(phi, 0)
    qc.barrier(label='Input')
    
    # Step 2: Create GHZ states for both hops
    # First GHZ: qubits [1,2,3]
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(1, 3)
    
    # Second GHZ: qubits [4,5,6]
    qc.h(4)
    qc.cx(4, 5)
    qc.cx(4, 6)
    qc.barrier(label='2 GHZ')
    
    # === FIRST HOP: Alice → Node1 ===
    print("\nFirst hop: Alice → Node1")
    
    # Inverse GHZ on [0,1,2]
    qc.cx(0, 2)
    qc.cx(0, 1)
    qc.h(0)
    
    # Measure
    qc.measure(0, 0)  # Alice
    qc.measure(1, 1)  # Charlie1
    
    # Corrections to q[3]
    # qc.z(3).c_if(cr[0], 1)
    # qc.x(3).c_if(cr[1], 1) 
    with qc.if_test((cr[0], 1)):
        qc.z(3)
    with qc.if_test((cr[1], 1)):
        qc.x(3)
    qc.barrier(label='Hop 1')
    
    # === SECOND HOP: Node1 → Bob ===
    print("Second hop: Node1 → Bob")
    
    # Inverse GHZ on [3,4,5]
    qc.cx(3, 5)
    qc.cx(3, 4)
    qc.h(3)
    
    # Measure
    qc.measure(3, 2)  # Node1
    qc.measure(4, 3)  # Charlie2
    
    # Corrections to q[6] (Bob)
    # qc.z(6).c_if(cr[2], 1)
    # qc.x(6).c_if(cr[3], 1)
    with qc.if_test((cr[2], 1)):
        qc.z(6)
    with qc.if_test((cr[3], 1)):
        qc.x(6)
    qc.barrier(label='Hop 2')
    
    print("\nCircuit created:")
    print(f"  Qubits: {qc.num_qubits}")
    print(f"  Depth: {qc.depth()}")
    print(f"  Gates: {sum(qc.count_ops().values())}")
    
    # Simulate
    qc.draw()
    qc.save_statevector()
    simulator = AerSimulator(method="statevector")
    result = simulator.run(qc).result()
    final_state = result.get_statevector()

    rho = DensityMatrix(final_state)
    rho_bob = partial_trace(rho, [0,1,2,3,4,5])

    target = Statevector([
        np.cos(theta),
        np.exp(1j * phi) * np.sin(theta)
    ])

    fidelity = state_fidelity(rho_bob, target)

    
    print(f"\nFidelity: {fidelity:.6f}")
    print(f"Status: {'✓ SUCCESS' if fidelity > 0.999 else '✗ FAILED'}")
    
    return qc, fidelity


# ============================================================================
# EXAMPLE 3: Test Multiple States
# ============================================================================

def test_multiple_states():
    """Test both protocols with different quantum states"""
    
    print("\n" + "="*70)
    print("TESTING MULTIPLE QUANTUM STATES")
    print("="*70)
    
    test_cases = [
        (0, 0, "|0⟩ state"),
        (np.pi/2, 0, "|+⟩ state"),
        (np.pi/2, np.pi, "|-⟩ state"),
        (np.pi/4, 0, "45° superposition"),
        (np.pi/3, np.pi/6, "General state"),
    ]
    
    print("\nSingle-Hop Results:")
    print("-" * 70)
    print(f"{'State':<25} {'θ':<10} {'φ':<10} {'Fidelity':<15}")
    print("-" * 70)
    
    for theta, phi, name in test_cases:
        qr = QuantumRegister(4)
        cr = ClassicalRegister(2)
        qc = QuantumCircuit(qr, cr)
        
        # Build circuit
        qc.ry(2*theta, 0)
        qc.rz(phi, 0)
        qc.h(1)
        qc.cx(1, 2)
        qc.cx(1, 3)
        qc.cx(0, 2)
        qc.cx(0, 1)
        qc.h(0)
        qc.measure(0, 0)
        qc.measure(1, 1)
        with qc.if_test((cr[0], 1)):
            qc.z(3)
        with qc.if_test((cr[1], 1)):
            qc.x(3)
        
        # Simulate
        qc.save_statevector()
        sim = AerSimulator(method='statevector')
        state = sim.run(qc).result().get_statevector()
        rho = DensityMatrix(state)
        rho = partial_trace(rho,[0,1,2])
        
        target = np.array([np.cos(theta), np.exp(1j*phi)*np.sin(theta)])
        fid = state_fidelity(rho, target)
        
        print(f"{name:<25} {theta:<10.4f} {phi:<10.4f} {fid:<15.6f}")
    
    print("\nDouble-Hop Results:")
    print("-" * 70)
    print(f"{'State':<25} {'θ':<10} {'φ':<10} {'Fidelity':<15}")
    print("-" * 70)
    
    for theta, phi, name in test_cases:
        qr = QuantumRegister(7)
        cr = ClassicalRegister(4)
        qc = QuantumCircuit(qr, cr)
        
        # Build circuit (double-hop)
        qc.ry(2*theta, 0)
        qc.rz(phi, 0)
        
        # Two GHZ states
        qc.h(1)
        qc.cx(1, 2)
        qc.cx(1, 3)
        qc.h(4)
        qc.cx(4, 5)
        qc.cx(4, 6)
        
        # First hop
        qc.cx(0, 2)
        qc.cx(0, 1)
        qc.h(0)
        qc.measure(0, 0)
        qc.measure(1, 1)
        if(cr[0] == 1):
            qc.z(3) 
        if(cr[1] == 1):
            qc.x(3)
        
        # Second hop
        qc.cx(3, 5)
        qc.cx(3, 4)
        qc.h(3)
        qc.measure(3, 2)
        qc.measure(4, 3)
        with qc.if_test((cr[2], 1)):
            qc.z(6)
        with qc.if_test((cr[3], 1)):
            qc.x(6)

        qc.draw()
        # Simulate
        qc.save_statevector()
        sim = AerSimulator(method='statevector')
        state = sim.run(qc).result().get_statevector()
        rho = DensityMatrix(state)
        rho = partial_trace(rho,[0,1,2,3,4,5])
        
        target = np.array([np.cos(theta), np.exp(1j*phi)*np.sin(theta)])
        fid = state_fidelity(rho, target)
        
        print(f"{name:<25} {theta:<10.4f} {phi:<10.4f} {fid:<15.6f}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    print("\n" + "╔" + "="*68 + "╗")
    print("║" + " "*10 + "QUANTUM CONTROLLED TELEPORTATION EXAMPLES" + " "*17 + "║")
    print("╚" + "="*68 + "╝")
    
    # Run examples
    single_qc, single_fid = single_hop_teleportation_example()
    double_qc, double_fid = double_hop_teleportation_example()
    test_multiple_states()
    
    # Summary
    print("\n" + "="*70)
    print("SUMMARY")
    print("="*70)
    print(f"\nSingle-Hop Teleportation:")
    print(f"  Qubits: {single_qc.num_qubits}")
    print(f"  Fidelity: {single_fid:.6f}")
    print(f"  Status: {'✓ Working perfectly' if single_fid > 0.999 else '✗ Needs fixing'}")
    
    print(f"\nDouble-Hop Teleportation:")
    print(f"  Qubits: {double_qc.num_qubits}")
    print(f"  Fidelity: {double_fid:.6f}")
    print(f"  Status: {'✓ Working perfectly' if double_fid > 0.999 else '✗ Needs fixing'}")
    
    print("\n" + "="*70)
    print("Done! Both protocols working correctly.")
    print("="*70 + "\n")



╔====================================================================╗
║          QUANTUM CONTROLLED TELEPORTATION EXAMPLES                 ║
╚====================================================================╝

SINGLE-HOP QUANTUM CONTROLLED TELEPORTATION

Input state: |φ⟩ = cos(0.785)|0⟩ + e^(i·0π)sin(0.785)|1⟩
           = 0.707|0⟩ + 0.707|1⟩

Circuit created:
  Qubits: 4
  Depth: 11
  Gates: 16

Fidelity: 1.000000
Status: ✓ SUCCESS

DOUBLE-HOP QUANTUM CONTROLLED TELEPORTATION

Input state: |φ⟩ = cos(0.785)|0⟩ + e^(i·0π)sin(0.785)|1⟩
           = 0.707|0⟩ + 0.707|1⟩

First hop: Alice → Node1
Second hop: Node1 → Bob

Circuit created:
  Qubits: 7
  Depth: 17
  Gates: 26

Fidelity: 1.000000
Status: ✓ SUCCESS

TESTING MULTIPLE QUANTUM STATES

Single-Hop Results:
----------------------------------------------------------------------
State                     θ          φ          Fidelity       
----------------------------------------------------------------------
|0⟩ state           